In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
from google.colab import files
uploaded=files.upload

In [14]:
spark = SparkSession.builder \
    .appName("ExpenseMonitoringETL") \
    .getOrCreate()

In [15]:
expense_df = spark.read.csv(
    "expenses.csv",
    header=True,
    inferSchema=True
)

In [16]:
expense_df = expense_df.withColumn(
    "date",
    to_date("date")
)

In [17]:
expense_df = expense_df.withColumn(
    "month",
    date_format("date", "yyyy-MM")
)

In [18]:
monthly_summary = (
    expense_df.groupBy("user_id", "month")
    .agg(
        sum("amount").alias("monthly_spend"),
        avg("amount").alias("average_spend"),
        count("*").alias("total_transactions")
    )
)

In [19]:
monthly_summary = monthly_summary.withColumn(
    "income",
    lit(10000)
)

In [22]:
monthly_summary = monthly_summary.withColumn(
    "savings",
    col("income") - col("monthly_spend")
)
monthly_summary.show()

+-------+-------+-------------+------------------+------------------+------+-------+
|user_id|  month|monthly_spend|     average_spend|total_transactions|income|savings|
+-------+-------+-------------+------------------+------------------+------+-------+
|      3|2026-06|         4900|1633.3333333333333|                 3| 10000|   5100|
|      3|2026-07|         3000|            1000.0|                 3| 10000|   7000|
|      1|2026-07|         4320|            1080.0|                 4| 10000|   5680|
|      1|2026-06|         3600|             900.0|                 4| 10000|   6400|
|      2|2026-06|         3500|1166.6666666666667|                 3| 10000|   6500|
|      2|2026-07|         6900|            2300.0|                 3| 10000|   3100|
+-------+-------+-------------+------------------+------------------+------+-------+



In [23]:
alerts = monthly_summary.filter(
    col("monthly_spend") > 7000
)
alerts.show()

+-------+-----+-------------+-------------+------------------+------+-------+
|user_id|month|monthly_spend|average_spend|total_transactions|income|savings|
+-------+-----+-------------+-------------+------------------+------+-------+
+-------+-----+-------------+-------------+------------------+------+-------+



In [24]:
monthly_summary.toPandas().to_csv(
    "monthly_expense_summary.csv",
    index=False
)

alerts.toPandas().to_csv(
    "savings_alert.csv",
    index=False
)